# Loops that grow, settle and tip {#sec-loops}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c03-loops.ipynb)

The waiting list in our running case more than doubled in ten years. Management now wants it brought down to a target. Before anyone argues about how, it helps to know the two forces at work: referrals that compound year on year, and a correction that pushes harder the further the list sits above its target. One is a **reinforcing loop**, a chain of causes that amplifies a change; the other is a **balancing loop**, a chain that opposes it. This chapter is about what each loop does alone, what happens when they share a stock, and how a structure can tip or lock in.

The figure after the setup cell sketches the case: the `Referral rate` feeds its own growth (loop R) and the `Waiting list` sets the extra capacity that drains it (loop B).

**Who it is for.** Anyone who will read or build feedback models. You will leave with the rules of thumb (doubling time, half-life, adjustment time) that let you check any model by eye. **Who can skip.** If you can already derive the logistic inflection point and explain path dependence, skim sections 1 to 3 and start at section 5. **Prerequisites.** Chapter 1, Why systems push back (stocks, flows and Euler steps) and chapter 2, From story to diagram (causal loop diagrams). **Time.** About three hours, with the exercises.

The data are synthetic, shaped like published Irish waiting-list counts; chapter 9, Data, calibration and uncertainty, explains how they were made and why.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 3
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| data:", wl.shape[0], "months,",
      wl.month.min().strftime("%b %Y"), "to", wl.month.max().strftime("%b %Y"))

In [ ]:
#| fig-alt: Sketch of two stocks. Referral rate grows through a reinforcing loop R and feeds Referrals into the Waiting list; the list drives extra capacity and so Treatments through a balancing loop B.
from matplotlib.patches import FancyArrowPatch, Rectangle


def draw_case(ax):
    """The chapter's case as a stock-and-flow sketch with its two loops."""
    def box(x, y, w, label):
        ax.add_patch(Rectangle((x, y), w, 0.2, fill=False, lw=2))
        ax.text(x + w / 2, y + 0.1, label, ha="center", va="center")

    def arrow(a, b, pipe=False, rad=0.0):
        style = dict(arrowstyle="-|>", mutation_scale=18 if pipe else 12,
                     lw=4 if pipe else 1.2, color="0.35" if pipe else "#0072B2",
                     connectionstyle=f"arc3,rad={rad}")
        ax.add_patch(FancyArrowPatch(a, b, **style))

    box(0.40, 0.60, 0.22, "Waiting list")
    box(0.12, 0.08, 0.22, "Referral rate")
    arrow((0.10, 0.70), (0.40, 0.70), pipe=True)          # Referrals
    arrow((0.62, 0.70), (0.92, 0.70), pipe=True)          # Treatments
    arrow((-0.10, 0.18), (0.12, 0.18), pipe=True)         # Referral growth
    ax.text(0.25, 0.76, "Referrals", ha="center")
    ax.text(0.77, 0.76, "Treatments", ha="center")
    ax.text(0.01, 0.10, "Referral growth", ha="center", fontsize=9)
    arrow((0.23, 0.28), (0.25, 0.68))                     # rate -> Referrals
    arrow((0.18, 0.08), (0.01, 0.16), rad=-0.6)           # rate -> growth
    ax.text(0.09, 0.32, "R", fontsize=15, weight="bold", ha="center")
    ax.text(0.82, 0.27, "Extra capacity\n(gap to target)", ha="center")
    arrow((0.56, 0.60), (0.68, 0.33), rad=0.2)            # list -> capacity
    arrow((0.80, 0.40), (0.78, 0.68))                     # capacity -> out
    ax.text(0.66, 0.48, "B", fontsize=15, weight="bold", ha="center")
    ax.set_xlim(-0.12, 1.0)
    ax.set_ylim(0.0, 0.85)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(7, 3.2))
draw_case(ax)
ax.set_title("The chapter's case: a growing inflow and a target")
plt.show()

Figure: The chapter's case. Thick arrows are flows of patients; thin arrows carry information. R marks the reinforcing loop on the referral rate, B the balancing loop that adds capacity when the list is above target.

**Learning outcomes.** By the end of the chapter you can:

1. **build** growth, decay and goal-seeking models and **predict** their doubling time, half-life or adjustment time before running them (Try it 1 to 3);
2. **use** ceilings and floors to keep a stock inside physical limits without breaking conservation (Try it 4);
3. **diagnose** which loop dominates a stock at each moment from its net flow (Try it 5);
4. **explain** S-curves, tipping points and lock-in with model runs, and **recognise** them in real systems (Try it 6 and 7);
5. **fit** one growth parameter to the waiting-list data and **justify** what a target policy can and cannot achieve (Try it 8 and the mini-project).

## 1. Growth: a stock that feeds its own inflow

The simplest reinforcing loop is a stock whose inflow is a fixed fraction of itself. More people mean more births; more births mean more people. The **Population Growth** molecule (`growth`), after Hines, *Molecules of Structure* [@hines_molecules], has births and deaths both proportional to the stock:

$$
\frac{dP}{dt} = b\,P - d\,P = (b - d)\,P, \qquad P(t) = P_0\,e^{(b-d)t}.
$$

When the net fractional growth rate $g = b - d$ is constant, the stock doubles every $\ln 2 / g$ time units. Since $\ln 2 \approx 0.69$, the **rule of 70** says: doubling time is about 70 divided by the growth rate in percent. Our version types the rates in percent a year (`Birth rate`, `Death rate`), as its sliders show them, and divides by 100 inside the flows.

In [ ]:
g = lb.molecule("growth")
print(g.equations())
pop = g.run()["Population"]          # births 5%, deaths 2% a year
exact = 100 * np.exp(0.03 * 60)
print(f"\nyear 60: run {pop.iloc[-1]:.1f}, exact {exact:.1f}")
print(f"doubling: rule of 70 gives {70 / 3:.1f} years, exact "
      f"{np.log(2) / 0.03:.2f}, run passes 200 in year "
      f"{pop.index[pop >= 200][0]}")
for dt in (1, 0.5, 0.1, 0.01):
    end = g.run(dt=dt)["Population"].iloc[-1]
    print(f"dt = {dt:<4}: year 60 is {end:6.1f}, "
          f"{100 * (end - exact) / exact:+.2f}% off")

With births of 5% and deaths of 2% a year, the net rate is 3%. The run passes 200 in year 23.2, against an exact doubling time of 23.10 years and a rule-of-70 estimate of 23.3. Euler's method always comes out a little low for growth, because each step uses the growth rate at the start of the step. With a step `dt` of one year it is 2.61% short after 60 years; with `dt = 0.1` the error is 0.27%.

A population growing by about 2.5% a year, close to Nigeria's rate over recent decades, doubles in roughly 70 / 2.5 = 28 years. The same arithmetic applies to referrals, loans or infections: anything whose growth is proportional to its own size.

In [ ]:
#| fig-alt: "Three population paths over 60 years: one rising ever faster, one flat at 100, one falling towards zero."
fig, ax = plt.subplots()
for label in ("Growing", "Stable", "Declining"):
    g.run(preset=label)["Population"].plot(ax=ax, label=label)
ax.set(title="Population Growth: three presets",
       xlabel="time (years)", ylabel="population (people)")
ax.legend()
plt.show()

Figure: Population Growth with three presets. When births exceed deaths the loop amplifies; when deaths exceed births the same structure shrinks the stock; when they are equal the stock stays put.

**Try it.**

- *Check.* Raise the birth rate to 8% (deaths stay at 2%). Use the rule of 70 to predict the doubling time, then run it.
- *Apply.* Run the 8% case with a time step of one month (`dt=1/12`). How far is year 60 from the exact value?
- *Extend.* Section 8 finds that referrals in the case grow by a fraction 0.0008 a month. What doubling time does that imply?

<details><summary>Worked answer</summary>

The net rate is 6%, so the rule of 70 predicts 11.7 years; the exact value is 11.55 and the run doubles in year 11.6. With a monthly step the run reaches 3,627 in year 60 against an exact 3,660, about 0.9% low: at fast growth a coarse step matters more. A rate of 0.0008 a month doubles in 72.2 years, so growth this slow hardly shows over a decade, yet it still compounds.
</details>

In [ ]:
fast = g.run(birth_rate=8)["Population"]
print("doubles in year", fast.index[fast >= 200][0],
      "| exact", round(np.log(2) / 0.06, 2))
monthly = g.run(birth_rate=8, dt=1 / 12)["Population"]
print(f"dt of one month, year 60: {monthly.iloc[-1]:.0f} against "
      f"exact {100 * np.exp(0.06 * 60):.0f}")
g_month = 0.0008                      # fraction per month
print(f"doubling time at {g_month} a month: "
      f"{np.log(2) / g_month / 12:.1f} years")

## 2. Decay, going to zero and residence time

Turn the loop around and the outflow is proportional to the stock. A full stock drains fast; a nearly empty one drains slowly. The **Go To Zero** molecule (`goToZero`), after Hines, *Molecules of Structure* [@hines_molecules], states the rule behind every such drain: the action that would remove a quantity over a time $\tau$ is the quantity divided by $\tau$.

$$
\text{Action} = \frac{S}{\tau}, \qquad S(t) = S_0\,e^{-t/\tau}, \qquad \text{half-life} = \tau \ln 2 \approx 0.69\,\tau .
$$

Our version runs a second stock beside the first, drained at the action's starting value and never updated, so you can see what the feedback adds. The **Decay** molecule (`decay`), after Hines, *Molecules of Structure* [@hines_molecules], is the same loop with nothing else attached; its time constant is called `Decay time`.

In [ ]:
#| fig-alt: "Two draining stocks from 100: a straight line that hits zero at month 5, and a curve that slows as it falls."
gz = lb.molecule("goToZero")
run = gz.run()
cur, const = run["Current value"], run["If action held constant"]
print("constant action empties the stock in month",
      const.index[const <= 1e-9][0])
print(f"proportional action: {cur.loc[5.0]:.1f} left after one time "
      f"constant (exact {100 * np.exp(-1):.1f}); half gone in month "
      f"{cur.index[cur <= 50][0]} (exact {5 * np.log(2):.2f})")
dec = lb.molecule("decay").run()["Stock"]   # decay time 5 years
print(f"Decay molecule: {dec.loc[5.0]:.1f} left after 5 years")

fig, ax = plt.subplots()
cur.loc[:20].plot(ax=ax, label="action proportional to what is left")
const.loc[:20].plot(ax=ax, label="action held at its first value")
ax.set(title="Go To Zero: two ways to drain a stock of 100",
       xlabel="time (months)", ylabel="stock (units)")
ax.legend()
plt.show()

Figure: Go To Zero with a time constant of 5 months. Holding the action at its first value empties the stock in exactly one time constant; acting in proportion to what is left never quite gets there.

The constant action empties the stock in month 5.0. The proportional one leaves 36.4 after one time constant (the exact value is 36.8, and Euler drains slightly too fast) and halves the stock in month 3.5, against an exact half-life of 3.47. That is the meaning of a **time constant**: how long the stock would take to vanish if the outflow stayed at its first value.

Add a steady inflow and the stock no longer goes to zero; it settles where inflow equals outflow. The **Residence Time** molecule (`residence`), after Hines, *Molecules of Structure* [@hines_molecules], feeds 8 people a month into a service where each stays 5 months on average. Ours starts empty, so you can watch it fill:

$$
\text{Outflow} = \frac{\text{In service}}{\tau}, \qquad \text{In service}^{*} = \text{Inflow} \times \tau .
$$

In [ ]:
res = lb.molecule("residence")
print(res.equations())
for tau in (2, 5, 10):
    d = res.run(avg_residence_time=tau)
    s, out = d["In service"], d["Outflow"]
    print(f"stay {tau:>2} months: {s.iloc[-1]:5.1f} in service at month 60,"
          f" outflow reaches 95% of inflow in month "
          f"{out.index[out >= 0.95 * 8][0]}, stock / outflow = "
          f"{s.iloc[-1] / out.iloc[-1]:.2f} months")

The stock settles at inflow times stay: 16, 40 and 80 people for stays of 2, 5 and 10 months (79.8 by month 60 for the longest, still closing). The outflow reaches 95% of the inflow after about three time constants. And in every case, stock divided by outflow gives back the average stay. Chapter 4, Delays, pipelines and memory, turns this into Little's law and applies it to the waiting list.

**Try it.**

- *Check.* The `With replenishment` preset of Go To Zero adds 8 a month to a stock drained over 5 months. Predict where it settles.
- *Apply.* In Residence Time, 12 people a month arrive and stay 5 months. How many are in service at steady state?
- *Extend.* Caffeine leaves the body of many adults with a half-life of around five hours. What time constant $\tau$ would you put in a Decay molecule?

<details><summary>Worked answer</summary>

Outflow equals inflow when stock / 5 = 8, so the stock settles at 40; the run ends at 40.00. With 12 a month, 12 × 5 = 60, as the run shows. A half-life of 5 hours means $\tau = 5 / \ln 2 = 7.21$ hours.
</details>

In [ ]:
rep = gz.run(preset="With replenishment")["Current value"]
print(f"with 8 a month coming in, the stock ends at {rep.iloc[-1]:.2f}")
d = res.run(inflow=12, avg_residence_time=5)
print(f"Residence Time, 12 a month, stay 5: {d['In service'].iloc[-1]:.1f}")
half_life = 5.0                       # hours
print(f"time constant for a {half_life}-hour half-life: "
      f"{half_life / np.log(2):.2f} hours")

## 3. Goal seeking: closing a gap

A balancing loop with a goal moves a stock towards that goal, closing a fixed fraction of the remaining gap per unit of time. The **Close Gap** molecule (`closegap`), after Hines, *Molecules of Structure* [@hines_molecules], is the simplest version:

$$
\frac{dA}{dt} = \frac{G - A}{T_{\text{adj}}}, \qquad A(t) = G - (G - A_0)\,e^{-t/T_{\text{adj}}} .
$$

The **adjustment time** $T_{\text{adj}}$ plays the part the time constant played in decay: the gap, not the stock, decays exponentially. Our version keeps the goal as a constant you can change; chapter 5, Decisions inside the model, lets the goal move.

In [ ]:
#| fig-alt: Three curves rising from 20 towards a dotted goal line at 60, faster for shorter adjustment times.
cg = lb.molecule("closegap")
print(cg.equations())
fig, ax = plt.subplots()
for label in ("Aggressive", "Reach goal", "Sluggish"):
    a = cg.run(preset=label)["Actual"]
    a.plot(ax=ax, label=f"{label}, adjustment time "
           f"{cg.preset(label)['at']} weeks")
    gap0 = 60 - 20
    t63 = a.index[a >= 60 - gap0 * np.exp(-1)][0]
    print(f"{label:<10}: 63% of the gap closed in week {t63}, "
          f"95% in week {a.index[a >= 60 - 0.05 * gap0][0]}")
ax.axhline(60, color="grey", lw=1, ls=":")
ax.set(title="Close Gap: one goal, three adjustment times",
       xlabel="time (weeks)", ylabel="actual (units)")
ax.legend()
plt.show()

Figure: Close Gap with adjustment times of 1.5, 5 and 15 weeks. The shape is the same; only the time scale changes.

Each run closes 63% of the gap after one adjustment time (weeks 1.5, 5.0 and 15.0) and 95% after about three (weeks 4.4, 14.9 and 44.8). Goal seeking is how a thermostat, a stock controller or a central bank behaves. New Zealand's Reserve Bank, the first to adopt an explicit inflation target in 1990, steers the gap between inflation and its target in just this spirit, although with far more delays than this molecule has.

**Try it.**

- *Check.* Without running anything, when will the `Reach goal` run first be within 5% of the gap? Compare with the output above.
- *Apply.* Run Close Gap with `dt` equal to the adjustment time, 5 weeks. What happens in the first step?
- *Extend.* Now use `dt=7.5`, one and a half adjustment times. Describe the path and explain why it is wrong.

<details><summary>Worked answer</summary>

About three adjustment times, 15 weeks; the run says 14.9. With `dt` equal to the adjustment time, one Euler step closes the whole gap: the stock jumps from 20 to 60 and stays. With `dt=7.5` it jumps to 80, falls to 50, then rises to 65: an oscillation that the model does not contain. It is an artefact of a step longer than the adjustment time. Keep `dt` well below the shortest time constant in a model (a quarter or less).
</details>

In [ ]:
a = cg.run(goal=60, adjustment_time=5, dt=5)["Actual"]
print("dt = adjustment time:", a.loc[[0.0, 5.0, 10.0]].tolist())
b = cg.run(goal=60, adjustment_time=5, dt=7.5)["Actual"]
print("dt = 1.5 x adjustment time:", b.loc[[0.0, 7.5, 15.0, 22.5]].tolist())

## 4. Hard limits: ceilings and floors

Physical limits are balancing loops too, often hidden inside a `MIN` or `MAX`. The **Ceiling** molecule (`ceiling`), after Hines, *Molecules of Structure* [@hines_molecules], lets a stock fill at its desired rate until the headroom gets small, then throttles the inflow so the stock eases onto the cap. The **Floor** molecule (`floor`), after Hines, *Molecules of Structure* [@hines_molecules], is its mirror for a draining stock.

$$
\text{Inflow} = \max\!\Big(0,\ \min\big(\text{Desired inflow},\ \tfrac{\text{Ceiling} - S}{T_{\text{approach}}}\big)\Big), \qquad
\text{Drain} = \max\!\Big(0,\ \min\big(\text{Desired outflow},\ \tfrac{S - \text{Floor}}{T_{\text{approach}}}\big)\Big).
$$

Our versions use an `Approach time` for the final approach, rather than clipping the stock, so the flows and the stock always add up.

In [ ]:
#| fig-alt: "Left: two stocks rising from 10 and levelling at 100. Right: two stocks falling from 100 and levelling at 30."
ce, fl = lb.molecule("ceiling"), lb.molecule("floor")
print(ce.equations(), "\n")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4), sharex=True)
for label in ("Approach cap", "Hard clip"):
    s = ce.run(preset=label)["Stock"]
    s.loc[:20].plot(ax=a1, label=label)
    print(f"ceiling, {label:<12}: {s.loc[5.0]:.1f} in week 5, "
          f"{s.loc[10.0]:.2f} in week 10, max {s.max():.2f}")
for label in ("Settle on floor", "Hard floor"):
    s = fl.run(preset=label)["Stock"]
    s.loc[:20].plot(ax=a2, label=label)
    print(f"floor, {label:<15}: {s.loc[5.0]:.1f} in week 5, "
          f"min {s.min():.2f}")
a1.set(title="Ceiling at 100", xlabel="time (weeks)",
       ylabel="stock (units)")
a2.set(title="Floor at 30", xlabel="time (weeks)")
a1.legend(); a2.legend()
plt.tight_layout()
plt.show()

Figure: Ceiling (left) and Floor (right). A short approach time gives a sharp corner; a longer one rounds it off. Neither stock ever crosses its limit.

The ceiling run with a 2-week approach time is at 79.1 in week 5 and 98.39 in week 10; with 0.5 weeks it is at the cap by week 10. The floor runs never go below 30. Cape Town's water crisis of 2018 had a floor written into its plans: the city prepared to turn off household taps if its dams fell to 13.5% of capacity, and steep restrictions throttled the outflow as levels approached that mark.

Our running case already has a floor: `Treatments = MIN(Capacity, Waiting_list / Minimum_treatment_time)` can never treat more patients than are waiting.

**Try it.**

- *Check.* With the default Ceiling, what is the inflow when the stock is at 90?
- *Apply.* How long does the stock take to come within 1 unit of the cap for approach times of 0.5, 2 and 8 weeks?
- *Extend.* In one sentence, say why a hard clip on the stock itself (resetting it to the cap) would be a worse model than a throttled flow.

<details><summary>Worked answer</summary>

The headroom is 10, so the inflow is min(14, 10 / 2) = 5. The stock comes within 1 of the cap in weeks 6.9, 11.0 and 35.8. Resetting the stock would destroy material without any flow carrying it away, so the stock would no longer equal the sum of its flows; throttling the flow keeps the books balanced.
</details>

In [ ]:
print("inflow at a stock of 90, approach time 2:", min(14, (100 - 90) / 2))
for at in (0.5, 2, 8):
    s = ce.run(approach_time=at)["Stock"]
    print(f"approach time {at:>3} weeks: within 1 of the cap in week "
          f"{s.index[s >= 99][0]}")

## 5. Two loops in one stock: the S-curve and shifting dominance

Put a reinforcing and a balancing loop on the same stock and the behaviour depends on which one is stronger at the moment. The **Constrained Growth** molecule (`logistic`), after Hines, *Molecules of Structure* [@hines_molecules], grows at a fractional rate $r$ but loses a share of that growth that rises as the stock fills its **carrying capacity** $K$:

$$
\frac{dP}{dt} = r\,P - r\,P\,\frac{P}{K} = r\,P\left(1 - \frac{P}{K}\right).
$$

Our version writes the crowding term as a separate outflow, `Crowding deaths`, so that the two loops appear as two flows you can plot.

In [ ]:
#| fig-alt: "Left: an S-shaped population rising from 10 to 200 with a dotted line at year 16.4. Right: a hump of net growth against population, peaking at 100."
lg = lb.molecule("logistic")
print(lg.equations())
d = lg.run()
net = d["Growth"] - d["Crowding deaths"]
peak = net.idxmax()
print(f"\nnet growth peaks in year {peak} at {net.max():.2f} a year, "
      f"population {d.loc[peak, 'Population']:.1f} (theory: r K / 4 = "
      f"{0.18 * 200 / 4:.1f} at K / 2 = 100, in year "
      f"{np.log(200 / 10 - 1) / 0.18:.1f})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
d["Population"].plot(ax=a1, label="population")
a1.axvline(peak, color="grey", ls=":", lw=1)
a1.set(title="S-curve", xlabel="time (years)",
       ylabel="population (people)")
a2.plot(d["Population"], net, color="#D55E00", ls="--")
a2.axvline(100, color="grey", ls=":", lw=1)
a2.set(title="Net flow against the stock", xlabel="population (people)",
       ylabel="net growth (people/year)")
plt.tight_layout()
plt.show()

Figure: Constrained Growth. Left: the S-curve, with its inflection point marked. Right: net growth plotted against the stock is a hump; the stock climbs it, then descends.

Net growth peaks in year 16.4, at 9.00 people a year, when the population is 99.6: exactly as theory predicts ($rK/4 = 9.0$ at $K/2 = 100$). Below 100, a bigger stock means more net growth, so the reinforcing loop **dominates**. Above 100, a bigger stock means less net growth, and the balancing loop dominates. **Loop dominance** is simply which loop is in charge of the stock's behaviour right now; **shifting dominance** is the handover from one to another.

You can label dominance from a run alone. If the net flow and its own rate of change have the same sign, the stock is accelerating away and a reinforcing loop dominates; if their signs differ, it is slowing down and a balancing loop dominates. This is the idea behind the behavioural approach to loop dominance of @ford1999. The helper below does it for any stock.

In [ ]:
def dominance(stock, dt):
    """Label each moment by the loop type that dominates the stock.

    If the net flow and its rate of change have the same sign, the stock is
    accelerating away (a reinforcing loop dominates); opposite signs mean
    it is slowing towards rest (a balancing loop dominates).
    """
    v = np.gradient(stock.values, dt)
    a = np.gradient(v, dt)
    tag = np.where(v * a > 0, "reinforcing", "balancing")
    tag[np.abs(v) < 1e-9] = "at rest"
    return pd.Series(tag, index=stock.index)


dom = dominance(d["Population"], 0.1)
switch = dom.index[(dom != dom.shift()) & (dom.index > 0.5)]
print("dominance changes at year(s):", list(switch.round(1)))
print(dom.loc[[5.0, 15.0, 20.0, 40.0]])

The label switches once, in year 16.5, from reinforcing to balancing (one step after the peak, because the derivatives are worked out numerically).

Real systems show the same handover. Japan's population grew for most of the twentieth century, peaked around 2008 and has fallen since: as the population aged, the balancing loop of deaths overtook the reinforcing loop of births. Shifting dominance is also why early data can mislead, as the next exercise shows.

**Try it.**

- *Check.* Where does the theory put the inflection point for any logistic curve?
- *Apply.* Run carrying capacities of 200, 350 and 100,000. Compare the populations in years 5 and 10. Could you tell the limits apart from the first five years of data?
- *Extend.* For each capacity, when does dominance shift? The exact formula is $t^{*} = \ln(K/P_0 - 1)/r$ with $P_0 = 10$.

<details><summary>Worked answer</summary>

At half the carrying capacity, where net growth is largest. In year 5 the three runs are at 22.8, 23.5 and 24.4: practically the same. By year 10 they are at 47.9, 52.3 and 59.5. Early growth carries almost no information about the limit, which is why forecasts of an S-curve fitted to its early part are so uncertain (chapter 9 returns to this). Dominance shifts in years 16.4, 19.7 and 51.6; the formula gives 16.4, 19.7 and 51.6.
</details>

In [ ]:
for K in (200, 350, 100000):
    run = lg.run(carrying_capacity=K)
    p = run["Population"]
    n = run["Growth"] - run["Crowding deaths"]
    print(f"K = {K:>6}: year 5 {p.loc[5.0]:6.1f}, year 10 {p.loc[10.0]:6.1f}"
          f", dominance shifts in year {n.idxmax()}")

## 6. Diffusion and tipping points

New products, habits and diseases spread in much the same way. In the **Diffusion** molecule (`diffusion`), after Hines, *Molecules of Structure* [@hines_molecules], people adopt for two reasons: some on their own or through advertising (innovation, $p$), and others because they meet adopters (imitation, $q$). This is the structure of the model of @bass1969:

$$
\text{Adoption} = \left(\frac{p}{100} + \frac{q}{100}\,\frac{A}{N}\right) \times \text{Potential}, \qquad N = \text{Potential} + A .
$$

Imitation is a reinforcing loop; the shrinking pool of potential adopters is the balancing limit. Our version types $p$ and $q$ in percent a week.

In [ ]:
#| fig-alt: "Left: adopters over 60 weeks for three presets, two S-shaped and one rising fastest at the start. Right: adoption rates, two humps and one falling curve."
dif = lb.molecule("diffusion")
print(dif.equations())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
for label in ("Word-of-mouth", "Ad-driven", "Viral"):
    run = dif.run(preset=label)
    run["Adopters"].plot(ax=a1, label=label)
    run["Adoption"].plot(ax=a2, label=label)
    half = run.index[run["Adopters"] >= 500][0]
    top = run["Adoption"]
    print(f"{label:<14}: adoption peaks in week {top.idxmax():4.1f} at "
          f"{top.max():5.1f} a week; half adopted by week {half}")
a1.set(title="Adopters", xlabel="time (weeks)", ylabel="adopters (people)")
a2.set(title="Adoption rate", xlabel="time (weeks)",
       ylabel="adoption (people/week)")
a1.legend()
plt.tight_layout()
plt.show()

Figure: Diffusion with three presets. Strong imitation gives a hump of adoptions and an S-curve; weak imitation gives sales that are highest at launch and only fall.

Word of mouth peaks in week 15.3 at 50.1 adoptions a week; the viral preset peaks earlier and higher, in week 12.1 at 90.0. The ad-driven product sells most at launch (its peak, 40.5 a week, comes in week 2.5) and has no real S. Norway's car market shows the S at national scale: battery-electric cars went from a few percent of new sales in the early 2010s to about four in five by 2022, with the steepest rise in the middle years.

A **tipping point** is a threshold where the dominant loop changes because of a small change in a parameter or a state. Add drop-out to diffusion (adopters give up after 10 weeks on average) and word of mouth must outrun forgetting. Each adopter recruits about $q/100 \times 10$ new adopters over their time as an adopter; when that number is below 1, the idea dies out. The helper functions make the spec short to write.

In [ ]:
def stock(i, name, init, units, non_negative=True):
    """A stock for a model spec."""
    return {"id": i, "name": name, "init": str(init), "units": units,
            "nonNegative": non_negative, "x": 0, "y": 0}


def flow(i, name, eq, units, src=None, dst=None):
    """A flow for a model spec, from stock src to stock dst."""
    f = {"id": i, "name": name, "eq": eq, "units": units, "x": 0, "y": 0}
    if src:
        f["from"] = src
    if dst:
        f["to"] = dst
    return f


def var(i, name, eq, units):
    """A variable (a constant or an equation) for a model spec."""
    return {"id": i, "name": name, "eq": str(eq), "units": units,
            "x": 0, "y": 0}

In [ ]:
# A simple tipping point: word of mouth that has to beat forgetting.
# Adopters drop out after 10 weeks on average; each adopter recruits
# Imitation_q / 100 of the non-adopters they meet each week.
dif_drop = lb.Model.from_spec({
    "name": "Diffusion with drop-out", "timeUnit": "week", "T": 150,
    "dt": 0.1,
    "stocks": [stock("p", "Potential", 990, "people"),
               stock("a", "Adopters", 10, "people")],
    "flows": [flow("ad", "Adoption", "Imitation_q / 100 * Adopters * "
                   "Potential / (Potential + Adopters)", "people/week",
                   "p", "a"),
              flow("dr", "Dropping out", "Adopters / Time_to_drop_out",
                   "people/week", "a", "p")],
    "vars": [var("q", "Imitation q", 18, "%/week"),
             var("td", "Time to drop out", 10, "weeks")]})
for q in (6, 9, 11, 14, 20):
    end = dif_drop.run(imitation_q=q)["Adopters"].iloc[-1]
    print(f"imitation {q:>2}% a week (q x drop-out time = {q / 10:.1f}):"
          f" {end:6.1f} adopters after 150 weeks")

Below the threshold of one, adopters vanish (0.0 and 2.1 left after 150 weeks for 0.6 and 0.9); just above it, the idea survives and keeps spreading (32.4 at 1.1, 267.4 at 1.4, 500.0 at 2.0). Nothing in the structure changed: a parameter crossed the point where the reinforcing loop beats the balancing one.

Epidemics tip the same way. In Samoa, measles vaccination coverage fell sharply in 2018; in late 2019 an epidemic spread through the population and killed more than 80 people, most of them young children. Vaccination works by keeping the number of new infections per case below one.

**Try it.**

- *Check.* Turn imitation off in the original Diffusion molecule (`imitation_q=0`). What shape do adopters follow, and how many have adopted by weeks 10, 20 and 60?
- *Apply.* Using `dif_drop`, find the smallest even value of `imitation_q` that keeps more than 10 adopters after 150 weeks.
- *Extend.* Halve the drop-out time to 5 weeks. Predict the new threshold, then check.

<details><summary>Worked answer</summary>

Without imitation only the innovation channel is left, so adoption is a pure decay of the potential pool: 95, 181 and 451 adopters by weeks 10, 20 and 60, a goal-seeking curve with no S. With a 10-week drop-out, the first even $q$ that keeps more than 10 adopters is 12% a week (just above the threshold of 10%). With 5 weeks the threshold doubles to 20%, and the first even value that works is 22%. Shortening the time people stay engaged is as powerful as cutting the contact rate.
</details>

In [ ]:
no_q = dif.run(imitation_q=0)["Adopters"]
print("no imitation, adopters in weeks 10, 20, 60:",
      no_q.loc[[10.0, 20.0, 60.0]].round(0).tolist())
for td in (10, 5):
    ends = {q: dif_drop.run(imitation_q=q, time_to_drop_out=td)
            ["Adopters"].iloc[-1] for q in range(2, 31, 2)}
    first = min(q for q, e in ends.items() if e > 10)
    print(f"drop-out after {td:>2} weeks: first q that keeps more than 10 "
          f"adopters is {first}% a week")

## 7. Lock-in: when success breeds success

When two options compete for the same newcomers, and each becomes more attractive the bigger it already is, a reinforcing loop decides who wins. The **Market Share** molecule (`marketshare`), after Hines, *Molecules of Structure* [@hines_molecules], splits 20 new customers a month between products A and B by relative attractiveness, with customers leaving after 18 months on average:

$$
\text{Share of A} = \frac{A^{\phi}}{A^{\phi} + B^{\phi}}, \qquad \phi = 1 + 2 \times \text{Network effect}.
$$

Our version builds attractiveness from the installed base raised to a power set by `Network effect`; with a network effect of 0, the split is proportional to size and nothing changes.

In [ ]:
#| fig-alt: "A's share of users over 60 months for four network effects: flat at 52.5% with none, rising towards 78%, 96% and 98% with stronger effects."
ms = lb.molecule("marketshare")
print(ms.equations(), "\n")
fig, ax = plt.subplots()
for n in (0, 0.35, 0.7, 1):
    run = ms.run(network_effect=n)
    share = run["Product A"] / (run["Product A"] + run["Product B"])
    (100 * share).plot(ax=ax, label=f"network effect {n}")
    print(f"network effect {n:<4}: A has {100 * share.loc[24.0]:.1f}% in "
          f"month 24, {100 * share.iloc[-1]:.1f}% in month 60")
ax.set(title="Market Share: an early lead of 105 to 95",
       xlabel="time (months)", ylabel="A's share of users (%)")
ax.legend()
plt.show()

Figure: Market Share starting from 105 users for A and 95 for B. With no network effect the share stays where it began; with one, the early leader pulls away.

With no network effect, A keeps its starting 52.5% for ever. With a network effect of 0.7, its share is 72.5% by month 24 and 95.9% by month 60; nothing about A is better except that it started slightly ahead. That is **path dependence**: where the system ends depends on where it began. Plotting the share A wins against the share it has shows why.

In [ ]:
#| fig-alt: "Share of newcomers against share of users: a diagonal, and S-shaped curves that cross it at 0, 0.5 and 1."
x = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for n in (0, 0.35, 0.7):
    phi = 1 + 2 * n
    ax.plot(x, x**phi / (x**phi + (1 - x)**phi), label=f"network effect {n}")
ax.plot(x, x, color="grey", lw=1, ls=":", label="no change")
ax.set(title="Where is the market heading?",
       xlabel="A's share of users (fraction)",
       ylabel="A's share of newcomers (fraction)")
ax.legend(fontsize=8)
plt.show()
phi = 1 + 2 * 0.7
for s in (0.45, 0.5, 0.55):
    print(f"A holds {s:.0%} of users and wins "
          f"{s**phi / (s**phi + (1 - s)**phi):.1%} of newcomers")

Figure: Where the curve is above the diagonal, A's share grows; below it, A's share shrinks. Crossings are equilibria.

With a network effect, the curve crosses the diagonal at 0, at one half and at 1. At one half it is steeper than the diagonal: a 55% share wins 61.8% of newcomers and grows further, so a tie is unstable, like a ball on an upturned bowl. At 0 and 1 the curve is flatter than the diagonal, so those ends are stable. Once a market sits there, it is **locked in**.

@arthur1989 showed how small historical accidents, amplified by increasing returns, can lock an economy into one technology. Australia's railways are a vivid case: in the nineteenth century the colonies laid three different track gauges, and each network's growth made switching costlier. A standard-gauge line from Sydney to Perth opened only in 1970.

**Try it.**

- *Check.* With the default network effect, start the race at 100 each. What happens, and why?
- *Apply.* Start at 101 to 99, then at 95 to 105. What share does A end with?
- *Extend.* Draw 200 random starting splits (A's users normally distributed around 100 with a standard deviation of 5, B getting the rest of 200). How often does the early leader win?

<details><summary>Worked answer</summary>

At 100 each the share stays at exactly 50.0%: the system sits on its unstable equilibrium and nothing pushes it off. A 101 to 99 start ends with A at 87.5% after 60 months; 95 to 105 leaves A with 4.1%. Across 200 random starts, A wins 103 times and the early leader wins in all 200. In this deterministic model, the first move decides everything; in real markets, chance and strategy keep shuffling the lead until the loop takes hold.
</details>

In [ ]:
def final_share(a, b, **kw):
    run = ms.with_init("Product A", a).with_init("Product B", b).run(**kw)
    end = run.iloc[-1]
    return 100 * end["Product A"] / (end["Product A"] + end["Product B"])


for a, b in ((100, 100), (101, 99), (95, 105)):
    print(f"start {a} to {b}: A ends with {final_share(a, b):.1f}%")
# extend: chance at the start, 200 markets with a random first split
starts = rng.normal(100, 5, size=(200, 1))
wins = np.array([final_share(s[0], 200 - s[0]) > 50 for s in starts])
ahead = starts[:, 0] > 100
print(f"A wins {wins.sum()} of 200 random starts; the early leader wins "
      f"{(wins == ahead).sum()} of 200")

## 8. The running case: referrals that grow, a target that pulls back

Now the case. Chapter 1, Why systems push back, built the waiting list as a bathtub. Here we add the two loops from the opening figure.

- **Referrals grow (loop R).** The `Referral rate` is a stock that grows by a fraction of itself each month, a Population Growth molecule in hospital clothes: a larger, older catchment generates more referrals, and each year's level is the base for the next.
- **A target pulls the list down (loop B).** When the target policy is on, management adds `Extra capacity` in proportion to the gap between the list and its `Target list`, a Close Gap molecule acting on a flow: $\text{Extra capacity} = \max(0, (\text{Waiting list} - \text{Target}) / T_{\text{adj}})$, optionally capped by a ceiling.

The spec below writes the whole model in one dictionary, with the names and starting values used throughout the book.

In [ ]:
CASE = {
    "name": "Waiting list with a target", "timeUnit": "month",
    "T": 120, "dt": 0.25,
    "stocks": [stock("wl", "Waiting list", 32000, "patients"),
               stock("rr", "Referral rate", 4000, "patients/month")],
    "flows": [
        flow("ref", "Referrals", "Referral_rate", "patients/month",
             dst="wl"),
        flow("trt", "Treatments",
             "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
             "patients/month", src="wl"),
        flow("rg", "Referral growth",
             "Referral_rate * Fractional_referral_growth",
             "patients/month/month", dst="rr")],
    "vars": [
        var("frg", "Fractional referral growth", 0.0025, "1/month"),
        var("base", "Base capacity", 3950, "patients/month"),
        var("tgt", "Target list", 25000, "patients"),
        var("at", "Adjustment time", 12, "months"),
        var("on", "Target on", 0, "1 = yes, 0 = no"),
        var("mx", "Maximum extra capacity", 1e6, "patients/month"),
        var("ex", "Extra capacity", "Target_on * MIN(Maximum_extra_capacity,"
            " MAX(0, (Waiting_list - Target_list) / Adjustment_time))",
            "patients/month"),
        var("cap", "Capacity", "Base_capacity + Extra_capacity",
            "patients/month"),
        var("mtt", "Minimum treatment time", 0.5, "months")]}
case = lb.Model.from_spec(CASE)
print(case.equations())

Only the stock is observed, so we cannot see referral growth directly. We can, however, ask which growth rate makes the model's list follow the data before the 2020 shock, holding `Base capacity` at 3,950 patients a month. `run_batch` runs the whole grid at once.

In [ ]:
#| fig-alt: "Monthly waiting list 2014 to 2023: the data rise from 32,000 to 70,000 with a jump in 2020; the fitted model follows them until 2020 and then stays well below."
pre = wl[wl.month < "2020-03-01"]          # before the 2020 shock
months = np.arange(len(pre), dtype=float)   # 0 = January 2014
grid = np.arange(0, 0.0041, 0.0001)         # fraction per month
runs = case.run_batch(pd.DataFrame({"fractional_referral_growth": grid}),
                      T=months[-1])
sim = runs[runs.variable == "Waiting list"].pivot(
    index="time", columns="run", values="value").loc[months]
rmse = np.sqrt(((sim.values - pre.total.values[:, None]) ** 2).mean(axis=0))
best = grid[rmse.argmin()]
print(f"best fractional referral growth {best:.4f} a month "
      f"= {100 * ((1 + best) ** 12 - 1):.2f}% a year, RMSE {rmse.min():.0f}"
      f" patients")
case = case.with_params(fractional_referral_growth=best)

fig, ax = plt.subplots()
ax.plot(wl.month, wl.total, color="#000000", ls="-", lw=1, label="data")
base = case.run()
ax.plot(wl.month, base["Waiting list"].loc[np.arange(120.0)],
        color="#0072B2", ls="--", label="model, no target")
ax.axvline(pd.Timestamp("2020-03-01"), color="grey", ls=":", lw=1)
ax.set(title="Fitting referral growth to the list before March 2020",
       xlabel="month", ylabel="waiting list (patients)")
ax.legend()
plt.show()

Figure: The waiting list (solid) and the model with fitted referral growth and no target (dashed). The dotted line marks March 2020; only months before it were used.

The best fit is a fraction 0.0008 a month, 0.96% a year, with a root mean square error of 526 patients. Be careful what this means: the data identify the gap between referrals and capacity, not either one. If capacity were also growing, referrals would have to grow faster to give the same list. Chapter 9, Data, calibration and uncertainty, deals with that properly. After 2020 the model and the data part company because the model has no shock; that too is for later chapters.

Now switch the target on: 25,000 patients, an adjustment time of 12 months. We run twenty years so the long-run behaviour shows.

In [ ]:
#| fig-alt: "Waiting list over 240 months under three policies: rising steeply with no target, settling slightly above a dotted target line with an unlimited target policy, and falling then rising again when extra capacity is capped."
scen = {"no target": dict(),
        "target, no limit": dict(target_on=1),
        "target, extra capacity at most 300": dict(
            target_on=1, maximum_extra_capacity=300)}
out = {k: case.run(T=240, **kw) for k, kw in scen.items()}
fig, ax = plt.subplots()
for k, d in out.items():
    d["Waiting list"].plot(ax=ax, label=k)
    print(f"{k:<36}: list {d['Waiting list'].loc[24.0]:7.0f} at month 24,"
          f" {d['Waiting list'].loc[120.0]:7.0f} at 120,"
          f" {d['Waiting list'].loc[240.0]:7.0f} at 240")
ax.axhline(25000, color="grey", ls=":", lw=1)
ax.set(title="A target of 25,000 patients, twenty years ahead",
       xlabel="time (months from January 2014)",
       ylabel="waiting list (patients)")
ax.legend()
plt.show()

Figure: Three policies. Without a target the list compounds upwards; an unlimited target holds it near, but above, 25,000; a capped target works at first and then loses.

Without a target the list reaches 61,742 at month 120 and 142,236 at month 240. With an unlimited target it is 29,934 at month 120: close to 25,000, but never on it. The reason is the **steady-state error** of a pure gap-closing rule. Extra capacity exists only while there is a gap, so the gap must stay large enough to pay for the referrals that base capacity cannot treat:

$$
\text{Waiting list} - \text{Target} \approx T_{\text{adj}} \times (\text{Referrals} - \text{Base capacity}).
$$

In [ ]:
d = out["target, no limit"]
err = d["Waiting list"] - 25000
pred = 12 * (d["Referrals"] - 3950)       # adjustment time x shortfall
print(pd.DataFrame({"gap to target": err, "12 x (referrals - base)": pred})
      .loc[[60.0, 120.0, 240.0]].round(0))

The formula predicts the gap well: 2,960 patients against an actual 2,525 at month 60, and 10,759 against 10,206 at month 240. It slightly overstates it because the list is still rising, so part of the shortfall goes into growth rather than into the gap. As referrals compound, the gap the rule tolerates compounds with them.

With extra capacity capped at 300 patients a month, the same policy works at first and then fails. Label the list's dominant loop in both target runs and watch its net inflow.

In [ ]:
for k in ("target, no limit", "target, extra capacity at most 300"):
    d = out[k]
    dom = dominance(d["Waiting list"], 0.25)
    switch = dom.index[(dom != dom.shift()) & (dom.index > 1)]
    net = (d["Referrals"] - d["Treatments"]).loc[[60.0, 120.0, 240.0]]
    print(f"{k}: dominance shifts in month(s) {list(switch)}; net inflow "
          f"at months 60, 120, 240 = {net.round(1).tolist()} patients/month")
ex = out["target, extra capacity at most 300"]["Extra capacity"]
print("the cap binds again from month", ex[(ex.index > 24) & (ex >= 300)]
      .index[0])

In both runs the list falls while the balancing loop dominates, bottoms out (month 32 without a cap, month 36.75 with one) and then rises: from there on, the compounding referrals set the direction. The difference is in how hard the balancing loop can still push back. Without a cap the net inflow stays small, 36.3 to 46.1 patients a month. With the cap, extra capacity hits its ceiling again in month 87.75; from then on the balancing loop is cut, and the net inflow climbs from 153.0 patients a month at month 120 to 596.6 at month 240.

That is shifting dominance and a tipping point in one run. A policy that "worked" for years can stop working without anyone changing it, simply because the demand it fights kept compounding past a limit.

**Try it.**

- *Check.* Using the formula above, would a shorter adjustment time make the gap at month 120 bigger or smaller?
- *Apply.* Run the unlimited target with adjustment times of 6, 12 and 24 months and report the list at month 120.
- *Extend.* How much extra capacity would management need at month 120 just to hold the list steady, with no target at all?

<details><summary>Worked answer</summary>

Smaller: the gap is proportional to the adjustment time. The runs give 27,592, 29,934 and 33,931 at month 120 for 6, 12 and 24 months. Holding the list steady needs referrals minus base capacity, 453 patients a month at month 120, which is more than the 300 cap: so the capped policy has to fail eventually. A rule that also replaced the growth in referrals, rather than only reacting to the gap, would not have this error; chapter 5, Decisions inside the model, builds one.
</details>

In [ ]:
for at in (6, 12, 24):
    e = case.run(T=120, target_on=1, adjustment_time=at)
    print(f"adjustment time {at:>2} months: list at month 120 = "
          f"{e['Waiting list'].iloc[-1]:,.0f}")
need = case.run(T=120)["Referrals"].iloc[-1] - 3950
print(f"extra capacity needed at month 120 to hold the list: {need:.0f}")

## Check it

Two habits catch most mistakes in models like these.

1. **Halve the time step.** If halving `dt` changes the answer noticeably, the step is too long for the fastest loop in the model.
2. **Balance the books.** A stock must equal its starting value plus the sum of its net flow times `dt`. If it does not, something (often a hard clip) is creating or destroying material.

In [ ]:
d = case.run(target_on=1)
for dt in (0.5, 0.25, 0.125):
    x = case.run(target_on=1, dt=dt)["Waiting list"].loc[120.0]
    print(f"dt = {dt:<5}: list at month 120 = {x:,.1f}")
inflow = (d["Referrals"] - d["Treatments"]).iloc[:-1].sum() * 0.25
change = d["Waiting list"].iloc[-1] - d["Waiting list"].iloc[0]
print(f"sum of net flow x dt = {inflow:,.1f}, change in stock = {change:,.1f}")

The list at month 120 changes by less than 1 patient between `dt` of 0.5, 0.25 and 0.125 months, and the net flow summed over the run equals the change in the stock (-2,066.2 patients either way). The model passes both checks.

## Running case

This chapter added two loops to the waiting list of chapter 1:

| Element | What it is | Value |
|---|---|---|
| `Referral rate` (stock) | referrals that grow by a fraction of themselves | starts at 4,000 patients/month |
| `Fractional referral growth` | fitted to the list before March 2020 | 0.0008 per month (0.96% a year) |
| `Extra capacity` | added in proportion to the gap to `Target list` | target 25,000 patients, adjustment time 12 months |
| `Maximum extra capacity` | a ceiling on that response | unlimited by default; 300 patients/month in the scenario |

Chapter 4, Delays, pipelines and memory, splits the list into waiting-time bands and adds the delays between what happens and what people perceive.

## Tell the decision maker

- A target enforced only by reacting to the gap will hold the list **above** the target, by roughly the adjustment time multiplied by the shortfall in capacity. With referrals growing, that excess grows too: about 4,900 patients after ten years in our run.
- If extra capacity has a ceiling below the eventual shortfall, the policy works for a while and then fails. In our run the ceiling binds again after about seven years, and from then on the list grows faster every year, with no change in management effort.
- To hit a target, capacity has to track the growth in referrals as well as close the gap. Ask for a forecast of referrals, not only a target.

## Going further

- **Loop dominance analysis** has formal tools beyond the behavioural label used here: eigenvalue elasticity and pathway participation methods measure each loop's contribution to the behaviour. @ford1999 is a readable start.
- **Path dependence with noise.** Replace the deterministic split in Market Share with random arrivals (a Pólya urn) and run thousands of histories with `run_batch` or numpy: the structure decides that someone wins; chance decides who.
- The molecules of this chapter can be explored interactively in the LoopBricks app, for example the [Market Share molecule](https://waelrash1.github.io/loopbricks/#/m/marketshare).

## Mini-project

Give the case a smarter decision rule and test it against the data.

1. Add a `Capacity growth` stock to the case (another Population Growth molecule) and fit both growth rates to the list before March 2020 with a two-dimensional grid. Plot the error surface. Is there one best pair, or a valley of equally good pairs? What does that say about what the data can tell you?
2. Change the target rule to $\text{Extra capacity} = \max(0, (\text{Waiting list} - \text{Target}) / T_{\text{adj}} + \text{Referrals} - \text{Base capacity})$, so it also replaces the shortfall. Show that the steady-state error disappears, and report the extra capacity this needs at month 120.
3. Write a half-page note for a hospital manager comparing the two rules, with one figure.

## Further reading

- @sterman2000, chapters 8 to 10, treats first-order systems, S-shaped growth and path dependence in depth.
- @meadows2008 gives a non-technical account of reinforcing and balancing loops and of shifting dominance.
- @hines_molecules is the source of the molecules used here.
- @bass1969 on diffusion, @arthur1989 on lock-in and @ford1999 on loop dominance are the classic papers behind sections 5 to 7.